# Tiền xử lý dữ liệu Instacart — Customer Features

Notebook này thực hiện **pipeline tiền xử lý dữ liệu** từ bộ dữ liệu Instacart:

1. **Tải dữ liệu** từ 4 bảng CSV chính (`products`, `prior`, `departments`, `orders`).
2. **Lọc dữ liệu** theo 7 nhóm hàng chiếm ~80% đơn hàng.
3. **Ghép dữ liệu** tạo bảng giao dịch chi tiết `prior_orders`.
4. **Tính toán 7 đặc trưng hành vi** khách hàng.
5. **Xuất** bảng `customer_features.csv` (206,209 khách hàng × 8 cột) cho các bước phân tích tiếp theo.

## 1. Import thư viện

| Thư viện | Mục đích |
|----------|----------|
| `pathlib.Path` | Xử lý đường dẫn tệp tin |
| `numpy` | Tính toán số học trên mảng |
| `pandas` | Đọc, xử lý và phân tích dữ liệu dạng bảng |
| `scipy.sparse` | Làm việc với ma trận thưa thớt (sparse matrix) — hữu ích khi tạo ma trận giao dịch lớn |
| `mlxtend.frequent_patterns` | Khai thác mẫu thường xuyên (`fpgrowth`) và sinh luật kết hợp (`association_rules`) |

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import scipy.sparse as sp
from mlxtend.frequent_patterns import fpgrowth, association_rules

pd.set_option('display.max_colwidth', None)
pd.set_option('display.max_rows', 20)

## 2. Tải dữ liệu từ các tệp CSV

Tải 4 bảng dữ liệu chính từ bộ dữ liệu Instacart:

| Tệp CSV | DataFrame | Mô tả | Các cột sử dụng |
|----------|-----------|--------|------------------|
| `products.csv` | `products` | Danh sách 49,688 sản phẩm | `product_id`, `product_name`, `department_id` |
| `order_products__prior.csv` | `prior` | 32,434,489 dòng — mỗi dòng là 1 sản phẩm trong 1 đơn hàng prior | Tất cả các cột |
| `departments.csv` | `departments` | 21 nhóm hàng (departments) | `department_id`, `department` |
| `orders.csv` | `orders` | 3,421,083 đơn hàng | Tất cả các cột |

> **Lưu ý:** Bảng `prior` load toàn bộ các cột (bao gồm `reordered`) vì cần dùng để tính `reorder_rate` ở các bước sau.

In [2]:
products = pd.read_csv('Data/products.csv', usecols=['product_id', 'product_name', 'department_id'])
prior = pd.read_csv('Data/order_products__prior.csv')
departments = pd.read_csv('Data/departments.csv')

print(f'Số sản phẩm: {len(products):,}')
print(f'Số dòng prior: {len(prior):,}')
print(f'Số departments: {len(departments):,}')
orders = pd.read_csv('Data/orders.csv')
print(f'Số đơn hàng: {len(orders):,}')
print(orders.head())

Số sản phẩm: 49,688
Số dòng prior: 32,434,489
Số departments: 21
Số đơn hàng: 3,421,083
   order_id  user_id eval_set  order_number  order_dow  order_hour_of_day  \
0   2539329        1    prior             1          2                  8   
1   2398795        1    prior             2          3                  7   
2    473747        1    prior             3          3                 12   
3   2254736        1    prior             4          4                  7   
4    431534        1    prior             5          4                 15   

   days_since_prior_order  
0                     NaN  
1                    15.0  
2                    21.0  
3                    29.0  
4                    28.0  


## 3. Lọc dữ liệu theo 7 nhóm hàng hàng đầu

7 nhóm hàng chiếm ~80% số đơn hàng (theo dashboard):

| STT | Department | Loại hàng |
|-----|-----------|----------|
| 1 | produce | Rau củ quả tươi |
| 2 | dairy eggs | Sữa và trứng |
| 3 | snacks | Đồ ăn nhẹ |
| 4 | beverages | Đồ uống |
| 5 | frozen | Đông lạnh |
| 6 | pantry | Thực phẩm khô/đóng hộp |
| 7 | bakery | Bánh mì/bánh ngọt |

**Các bước xử lý:**
1. Ghép `products` với `departments` trên `department_id` → `products_dept`
2. Lọc sản phẩm thuộc 7 nhóm → `products_top` (26,656 sản phẩm)
3. Lọc `prior` chỉ giữ sản phẩm thuộc 7 nhóm → `prior_top` (25,759,782 dòng, giữ ~79.4%)

In [3]:
# 7 nhóm hàng chiếm 80% số đơn (theo dashboard):
top_departments = [
    'produce', 'dairy eggs', 'snacks', 'beverages', 'frozen', 'pantry', 'bakery'
]

# Join products với departments để lấy department cho từng sản phẩm
products_dept = products.merge(departments, on='department_id', how='left')

# Lọc sản phẩm thuộc 7 nhóm hàng này
products_top = products_dept[products_dept['department'].isin(top_departments)]

# Lọc prior chỉ giữ các sản phẩm thuộc 7 nhóm này
prior_top = prior[prior['product_id'].isin(products_top['product_id'])].copy()

print(f'Số sản phẩm thuộc 7 nhóm: {len(products_top):,}')
print(f'Số dòng prior sau lọc: {len(prior_top):,}')

Số sản phẩm thuộc 7 nhóm: 26,656
Số dòng prior sau lọc: 25,759,782


## 4. Ghép dữ liệu và thêm cột `is_organic`

Ô này thực hiện 3 bước quan trọng để chuẩn bị dữ liệu cho việc tính toán đặc trưng khách hàng:

1. **Tạo cột `is_organic`:** Kiểm tra tên sản phẩm có chứa từ "organic" hay không (1 = hữu cơ, 0 = không).
2. **Ghép `prior` + `products`:** Thêm cột `is_organic` vào mỗi dòng giao dịch qua `product_id`.
3. **Ghép với `orders` → `prior_orders`:** Thêm `user_id`, `order_dow`, `days_since_prior_order` → tạo bảng giao dịch chi tiết.

Cấu trúc bảng `prior_orders` sau ghép:

| Cột | Nguồn | Ý nghĩa |
|-----|-------|--------|
| `order_id` | `prior` | Mã đơn hàng |
| `product_id` | `prior` | Mã sản phẩm |
| `reordered` | `prior` | Sản phẩm có được mua lại không (0/1) |
| `is_organic` | `products` | Sản phẩm hữu cơ hay không (0/1) |
| `user_id` | `orders` | Mã khách hàng |
| `order_dow` | `orders` | Ngày trong tuần đặt hàng (0–6) |
| `days_since_prior_order` | `orders` | Số ngày kể từ đơn hàng trước |

In [14]:
# Merge prior với orders để có user_id, order_dow, days_since_prior_order, order_hour_of_day
prior_orders = prior.merge(orders[['order_id', 'user_id', 'order_dow', 'days_since_prior_order', 'order_hour_of_day']], on='order_id', how='left')

# Lọc chỉ prior orders
prior_orders = prior_orders[prior_orders['user_id'].notna()]

print('Data merged successfully')

Data merged successfully


## 5. Tính toán các đặc trưng khách hàng (Customer Features)

Từ bảng `prior_orders`, tính 9 đặc trưng hành vi cho mỗi `user_id`:

| STT | Đặc trưng | Công thức | Ý nghĩa | Vai trò trong phân cụm |
|-----|-----------|-----------|--------|-----------------------|
| 1 | `total_orders` | COUNT(order_id) | Tổng số đơn | Đo mức độ hoạt động |
| 2 | `average_days_between_orders` | AVG(days_since_prior_order) | Khoảng cách trung bình giữa các lần mua | Phân biệt mua thường xuyên vs thưa |
| 3 | `reorder_rate` | SUM(reordered) / COUNT(product_id) | Tỷ lệ mua lại | Nhận diện hành vi trung thành |
| 4 | `avg_basket_size` | total_items / total_orders | Trung bình số item mỗi đơn | Phân biệt mua ít và nhiều |
| 5 | `unique_products` | COUNT(DISTINCT product_id) | Số sản phẩm khác nhau | Đo mức độ đa dạng |
| 6 | `product_diversity` | unique_products / total_items | Tỷ lệ đa dạng sản phẩm | Phân biệt explore vs focus |
| 7 | `weekend_ratio` | weekend_orders / total_orders | % đơn cuối tuần (T7-CN) | Nhận diện hành vi theo ngày |
| 8 | `night_order_ratio` | night_orders / total_orders | % đơn ban đêm (22h–5h) | Phân biệt lifestyle |
| 9 | `organic_ratio_order` | SUM(is_organic) / COUNT(product_id) | Tỷ lệ mua organic | Phân biệt sở thích |

---

### 5.1 `total_orders` — Tổng số đơn hàng

Lọc đơn hàng có `eval_set == 'prior'`, nhóm theo `user_id` và đếm số `order_id`.

> **Ý nghĩa:** Phản ánh tần suất mua hàng. Giá trị cao = khách hàng trung thành, mua thường xuyên.

In [5]:
# total_orders
orders_prior = orders[orders['eval_set'] == 'prior']
total_orders = orders_prior.groupby('user_id')['order_id'].count().rename('total_orders')
print('total_orders computed')

total_orders computed


### 5.2 `average_days_between_orders` — Số ngày trung bình giữa các đơn hàng

Loại bỏ đơn hàng đầu tiên (`days_since_prior_order = NaN`), rồi tính trung bình theo `user_id`.

> **Ý nghĩa:** Phản ánh chu kỳ mua hàng. Giá trị thấp (5–7 ngày) = mua rất thường xuyên; giá trị cao (25–30 ngày) = mua theo tháng.

In [6]:
# average_days_between_orders
average_days_between_orders = prior_orders[prior_orders['days_since_prior_order'].notna()].groupby('user_id')['days_since_prior_order'].mean().rename('average_days_between_orders')
print('average_days_between_orders computed')

average_days_between_orders computed


### 5.3 `reorder_rate` — Tỷ lệ mua lại sản phẩm

**Công thức:** `reorder_rate = Σ(reordered) / Σ(total_items)`

> **Ý nghĩa:**
> - Gần 1: khách hàng hay mua lại sản phẩm cũ → hành vi ổn định, quen thuộc.
> - Gần 0: khách hàng hay thử sản phẩm mới → hành vi khám phá, đa dạng.

In [7]:
# reorder_rate
reorder_rate = (prior_orders.groupby('user_id')['reordered'].sum() / prior_orders.groupby('user_id')['product_id'].count()).rename('reorder_rate')
print('reorder_rate computed')

reorder_rate computed


### 5.4 `avg_basket_size` — Kích thước giỏ hàng trung bình

**Công thức:** `avg_basket_size = total_items / total_orders`

> **Ý nghĩa:**
> - Giá trị cao (15–20): mua nhiều sản phẩm mỗi lần → gia đình lớn hoặc mua sắm theo tuần.
> - Giá trị thấp (3–5): mua ít sản phẩm mỗi lần → mua bổ sung hoặc hộ gia đình nhỏ.

In [8]:
# avg_basket_size
total_items = prior_orders.groupby('user_id')['product_id'].count()
avg_basket_size = (total_items / total_orders).rename('avg_basket_size')
print('avg_basket_size computed')

avg_basket_size computed


### 5.5 `unique_products` — Số sản phẩm khác nhau đã mua

Dùng `nunique()` để đếm số sản phẩm **duy nhất** (không trùng) mà mỗi khách hàng đã mua.

> **Ý nghĩa:** Phản ánh mức độ đa dạng trong hành vi mua sắm. Giá trị cao = hay thử sản phẩm mới; giá trị thấp = chỉ mua vài sản phẩm quen thuộc.

In [9]:
# unique_products
unique_products = prior_orders.groupby('user_id')['product_id'].nunique().rename('unique_products')
print('unique_products computed')

unique_products computed


### 5.6 `weekend_ratio` — Tỷ lệ đặt hàng cuối tuần

**Công thức:** `weekend_ratio = weekend_orders / total_orders`

Cuối tuần được xác định là `order_dow ∈ {0 (Chủ nhật), 6 (Thứ Bảy)}`. Dùng `fillna(0)` cho khách hàng không có đơn cuối tuần.

> **Ý nghĩa:** Phản ánh thói quen mua sắm theo thời gian. Gần 1 = chủ yếu mua cuối tuần; gần 0 = chủ yếu mua ngày thường.

In [10]:
# weekend_ratio
weekend_orders = prior_orders[prior_orders['order_dow'].isin([0, 6])].groupby('user_id')['order_id'].nunique()
weekend_ratio = (weekend_orders / total_orders).fillna(0).rename('weekend_ratio')
print('weekend_ratio computed')

weekend_ratio computed


### 5.7 `organic_ratio_order` — Tỷ lệ sản phẩm hữu cơ

**Công thức:** `organic_ratio_order = Σ(is_organic) / Σ(total_items)`

> **Ý nghĩa:**
> - Gần 1: ưu tiên mua sản phẩm hữu cơ → quan tâm sức khỏe, sẵn sàng chi trả cao hơn.
> - Gần 0: ít mua sản phẩm hữu cơ → ưu tiên giá cả hoặc không quan tâm.

In [11]:
# organic_ratio_order
organic_ratio_order = (prior_orders.groupby('user_id')['is_organic'].sum() / prior_orders.groupby('user_id')['product_id'].count()).rename('organic_ratio_order')
print('organic_ratio_order computed')

organic_ratio_order computed


### 5.8 `product_diversity` — Tỷ lệ đa dạng sản phẩm

**Công thức:** `product_diversity = unique_products / total_items`

> **Ý nghĩa:** Phản ánh mức độ đa dạng trong hành vi mua sắm. Giá trị cao = hay thử sản phẩm mới (explore); giá trị thấp = chỉ mua vài sản phẩm quen thuộc (focus).

### 5.9 `night_order_ratio` — Tỷ lệ đặt hàng ban đêm

**Công thức:** `night_order_ratio = night_orders / total_orders`

Ban đêm được xác định là `order_hour_of_day ∈ {22, 23, 0, 1, 2, 3, 4, 5}`. Dùng `fillna(0)` cho khách hàng không có đơn ban đêm.

> **Ý nghĩa:** Phản ánh thói quen mua sắm theo thời gian. Gần 1 = chủ yếu mua ban đêm; gần 0 = chủ yếu mua ban ngày.

In [15]:
# product_diversity
total_items = prior_orders.groupby('user_id')['product_id'].count()
product_diversity = (unique_products / total_items).rename('product_diversity')
print('product_diversity computed')

# night_order_ratio
night_hours = [22, 23, 0, 1, 2, 3, 4, 5]
night_orders = prior_orders[prior_orders['order_hour_of_day'].isin(night_hours)].groupby('user_id')['order_id'].nunique()
night_order_ratio = (night_orders / total_orders).fillna(0).rename('night_order_ratio')
print('night_order_ratio computed')

product_diversity computed
night_order_ratio computed


## 6. Tạo bảng `customer_features`

Ghép 7 Series đặc trưng theo chiều ngang (`axis=1`) thành DataFrame hoàn chỉnh, rồi `reset_index()` để `user_id` thành cột bình thường.

**Kết quả mong đợi:** 206,209 khách hàng × 8 cột (`user_id` + 7 features).

In [16]:
# Tạo bảng customer_features
customer_features = pd.concat([
    total_orders,
    average_days_between_orders,
    reorder_rate,
    avg_basket_size,
    unique_products,
    product_diversity,
    weekend_ratio,
    night_order_ratio,
    organic_ratio_order
], axis=1).reset_index()

print(f'Bảng customer_features có {len(customer_features)} khách hàng')
print(customer_features.head())

Bảng customer_features có 206209 khách hàng
   user_id  total_orders  average_days_between_orders  reorder_rate  \
0        1            10                    20.259259      0.694915   
1        2            14                    15.967033      0.476923   
2        3            12                    11.487179      0.625000   
3        4             5                    15.357143      0.055556   
4        5             4                    14.500000      0.378378   

   avg_basket_size  unique_products  product_diversity  weekend_ratio  \
0         5.900000               18           0.305085           0.00   
1        13.928571              102           0.523077           0.00   
2         7.333333               33           0.375000           0.50   
3         3.600000               17           0.944444           0.20   
4         9.250000               23           0.621622           0.25   

   night_order_ratio  organic_ratio_order  
0                0.0             0.254237  
1 

## 7. Xuất dữ liệu ra CSV

Lưu bảng `customer_features` ra file `customer_features.csv` (`index=False` để không ghi cột index).

File này sẽ được sử dụng cho các bước phân tích tiếp theo như phân cụm khách hàng (K-Means, DBSCAN...) hoặc phân tích luật kết hợp.

In [17]:
customer_features.to_csv('customer_features.csv', index=False)